# CKD — XAI extension

The reusable XAI framework is applied to the pre-specified Random Forest model with 200 trees and `random_state=42`. Imputation and categorical encoding are fitted using training data only.

The framework generates explanations and assesses bootstrap-refit stability. Global permutation importance uses ROC AUC consistently across all five configurations, while bootstrap-refit stability uses negative log loss consistently across the four retrainable predictive configurations. Results are exported to `../../results/ckd`.



The exported global SHAP analysis uses SHAP 0.51.0 `TreeExplainer`.


## Data and model preparation


In [ ]:
import os

os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import warnings
from tqdm import TqdmWarning

warnings.filterwarnings(
    "ignore",
    message="IProgress not found.*",
    category=TqdmWarning,
)

import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name != "xai_extension":
    raise RuntimeError(
        "Run this notebook with the working directory set to "
        "thesis/src/xai_extension. Current directory: "
        f"{NOTEBOOK_DIR}"
    )

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
FRAMEWORK_DIR = PROJECT_ROOT / "src" / "xai_framework"
if str(FRAMEWORK_DIR) not in sys.path:
    sys.path.insert(0, str(FRAMEWORK_DIR))

from run_pipeline import run_pipeline

# Load and clean the data.
file_path = "../../data/ckd/kidney_disease.csv"
DATASET_NAME = "kidney_disease"
df = pd.read_csv(file_path)

df = df.replace(r'^\s+|\s+$', '', regex=True)
df = df.replace('?', np.nan)
df = df.replace('\t?', np.nan)
df = df.replace({'\tyes': 'yes', '\tno': 'no', ' yes': 'yes', ' no': 'no'})

possible_targets = ['classification', 'class', 'target', 'label']
target_col = next((c for c in possible_targets if c in df.columns), None)
if target_col is None:
    raise ValueError("Target column not found.")

df[target_col] = df[target_col].astype(str).str.strip().str.lower()
df[target_col] = df[target_col].replace({
    'ckd\t': 'ckd',
    'notckd': 'notckd',
    'not ckd': 'notckd'
})

df = df[df[target_col].isin(['ckd', 'notckd'])].copy()

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

X = X.drop(columns=['id'])

for col in X.columns:
    converted = pd.to_numeric(X[col], errors='coerce')
    if converted.notna().sum() == X[col].notna().sum():
        X[col] = converted

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

y = y.map({'ckd': 1, 'notckd': 0})

print("Shape:", X.shape)
print("Target counts:\n", y.value_counts())
print("Numeric:", numeric_cols)
print("Categorical:", categorical_cols)

# Create the stratified split before preprocessing.
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# Fit imputation using the training partition only.
rng = np.random.default_rng(42)

def random_sample_impute_train_test(train_s, test_s, rng_obj):
    """Impute missing values using only the observed training distribution.

    Args:
        train_s: Training series containing observed and possibly missing values.
        test_s: Held-out series to impute without using its observed distribution.
        rng_obj: Random-number generator providing a ``choice`` method.

    Returns:
        Copies of the training and test series with missing entries sampled
        from the observed training values.
    """
    train_s = train_s.copy()
    test_s = test_s.copy()

    observed = train_s.dropna().values
    if len(observed) == 0:
        return train_s, test_s

    train_missing = train_s.isna()
    if train_missing.sum() > 0:
        sampled_train = rng_obj.choice(observed, size=train_missing.sum(), replace=True)
        train_s.loc[train_missing] = sampled_train

    observed = train_s.dropna().values

    test_missing = test_s.isna()
    if test_missing.sum() > 0:
        sampled_test = rng_obj.choice(observed, size=test_missing.sum(), replace=True)
        test_s.loc[test_missing] = sampled_test

    return train_s, test_s

for col in numeric_cols:
    try:
        X_train[col], X_test[col] = random_sample_impute_train_test(X_train[col], X_test[col], rng)
    except Exception:
        mean_val = X_train[col].mean()
        X_train[col] = X_train[col].fillna(mean_val)
        X_test[col] = X_test[col].fillna(mean_val)

for col in categorical_cols:
    try:
        X_train[col], X_test[col] = random_sample_impute_train_test(X_train[col], X_test[col], rng)
    except Exception:
        mode_val = X_train[col].mode(dropna=True)
        if len(mode_val) > 0:
            fill_val = mode_val.iloc[0]
            X_train[col] = X_train[col].fillna(fill_val)
            X_test[col] = X_test[col].fillna(fill_val)

for col in numeric_cols:
    mean_val = X_train[col].mean()
    X_train[col] = X_train[col].fillna(mean_val)
    X_test[col] = X_test[col].fillna(mean_val)

for col in categorical_cols:
    mode_val = X_train[col].mode(dropna=True)
    if len(mode_val) > 0:
        fill_val = mode_val.iloc[0]
        X_train[col] = X_train[col].fillna(fill_val)
        X_test[col] = X_test[col].fillna(fill_val)

# Fit categorical encoding using the training partition only.

encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train[categorical_cols] = encoder.fit_transform(
    X_train[categorical_cols].astype(str)
)

X_test[categorical_cols] = encoder.transform(
    X_test[categorical_cols].astype(str)
)

Shape: (400, 24)
Target counts:
 classification
1    250
0    150
Name: count, dtype: int64
Numeric: ['age', 'bp', 'sg', 'al', 'su', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc']
Categorical: ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane']


In [ ]:
# Retain the leakage-controlled matrices and labels.

X_train_no_leakage = X_train.copy()
X_test_no_leakage = X_test.copy()

y_train_no_leakage = y_train.copy()
y_test_no_leakage = y_test.copy()

models_no_leakage = {
    "RF": RandomForestClassifier(n_estimators=200, random_state=42)
}

In [ ]:
# Fit the pre-specified CKD model.

best_model_name = "RF"
print("Pre-specified XAI model from source study:", best_model_name)


best_model = clone(models_no_leakage[best_model_name])
best_model.fit(X_train_no_leakage, y_train_no_leakage)


def train_model_fn(X_bootstrap, y_bootstrap):
    """Refit the fixed CKD estimator on one bootstrap training sample.

    Args:
        X_bootstrap: Bootstrap-resampled training feature matrix.
        y_bootstrap: Outcomes aligned with the bootstrap feature rows.

    Returns:
        A newly cloned and fitted estimator with the pre-specified CKD
        configuration.
    """
    model = clone(models_no_leakage[best_model_name])
    model.fit(X_bootstrap, y_bootstrap)
    return model

Pre-specified XAI model from source study: RF


## Execute the XAI pipeline


In [ ]:
PILOT = False

config = {
    "dataset_name": "ckd",

    "model": best_model,

    "X_train": X_train_no_leakage,
    "X_test": X_test_no_leakage,

    "y_train": y_train_no_leakage,
    "y_test": y_test_no_leakage,

    "feature_names": list(X_train_no_leakage.columns),

    # Repeated local explanations
    "local_cases_per_type": 2 if PILOT else 10,
    "lime_seeds": list(range(2 if PILOT else 20)),
    "lime_num_features": 5,
    "lime_stability_top_k": 3,

    "train_model_fn": train_model_fn,

    # Bootstrap-refit stability
    "stability_n_bootstrap": 10 if PILOT else 100,
    "stability_bootstrap_seed": 42,
    "stability_scoring": "neg_log_loss",
    "stability_n_repeats": 10,
    "stability_permutation_seed": 42,
    "stability_top_k": 3,

    # Ordinary global permutation analysis
    "scoring": "roc_auc",

    # Keep pilot and final results separate
    "output_dir": (
        "../../results/pilot_stability/ckd"
        if PILOT
        else "../../results/ckd"
    ),
}

experiment = run_pipeline(config)

Results exported to: ../../results/ckd


## XAI framework output sanity checks


In [ ]:
print("Dataset:", config["dataset_name"])
print("Model:", type(config["model"]).__name__)

print("X_train shape:", config["X_train"].shape)
print("X_test shape:", config["X_test"].shape)

print("y_train length:", len(config["y_train"]))
print("y_test length:", len(config["y_test"]))

print("Feature names:", len(config["feature_names"]))

Dataset: ckd
Model: RandomForestClassifier
X_train shape: (280, 24)
X_test shape: (120, 24)
y_train length: 280
y_test length: 120
Feature names: 24


### Input consistency


In [ ]:
assert config["X_train"].shape[1] == len(config["feature_names"])
assert config["X_test"].shape[1] == len(config["feature_names"])

assert len(config["X_train"]) == len(config["y_train"])
assert len(config["X_test"]) == len(config["y_test"])

print("Basic shape validation: OK")

Basic shape validation: OK


### Prediction-interface check


In [ ]:
model = config["model"]
X_test = config["X_test"]

if hasattr(model, "predict_proba"):
    probs = model.predict_proba(X_test)[:, 1]
    print("Probability range:", probs.min(), probs.max())

preds = model.predict(X_test)

print("Predictions:", np.unique(preds, return_counts=True))
print("Prediction test: OK")

Probability range: 0.0 1.0
Predictions: (array([0, 1]), array([45, 75]))
Prediction test: OK


### Framework results


In [ ]:
global_results = experiment.state.results["global"]
local_results = experiment.state.results["local"]
stability_results = experiment.state.results["stability"]

print(global_results.keys())
print(local_results.keys())

if stability_results is not None:
    print(stability_results.keys())
else:
    print("Stability: not available")

dict_keys(['native_importance', 'permutation_importance', 'shap_importance', 'shap_values', 'shap_raw_values', 'explainer', 'shap_explainer_class', 'rank_compare', 'rank_agreement'])
dict_keys(['cases', 'lime', 'lime_summary', 'lime_seed_stability'])
dict_keys(['raw', 'summary', 'rank_stability'])


### Global explanations


In [ ]:
native_df = global_results["native_importance"]
perm_df = global_results["permutation_importance"]
shap_df = global_results["shap_importance"]

print("Native:")
display(native_df.head())

print("Permutation:")
display(perm_df.head())

print("SHAP:")
display(shap_df.head())

Native:


,Feature,Importance,Method
11,sc,0.172794,tree_importance
2,sg,0.172209,tree_importance
14,hemo,0.151966,tree_importance
15,pcv,0.137196,tree_importance
3,al,0.073456,tree_importance


Permutation:


,Feature,Importance,Std
2,sg,0.011378,0.006472
11,sc,0.006681,0.003167
15,pcv,0.003311,0.002984
17,rc,0.000778,0.000140
3,al,0.000459,0.000434


SHAP:


,Feature,MeanAbsSHAP,MeanSHAP
0,sg,0.093746,-0.004480
1,sc,0.082862,-0.015533
2,hemo,0.078469,-0.003660
3,pcv,0.074007,-0.001781
4,al,0.043660,-0.000041


### Feature coverage


In [ ]:
if native_df is not None:
    print("Native features:", len(native_df))

print("Permutation features:", len(perm_df))
print("SHAP features:", len(shap_df))
print("Expected features:", len(config["feature_names"]))

Native features: 24
Permutation features: 24
SHAP features: 24
Expected features: 24


### Numerical integrity


In [ ]:
print("Permutation NaNs:", perm_df.isna().sum().sum())
print("SHAP NaNs:", shap_df.isna().sum().sum())

if native_df is not None:
    print("Native NaNs:", native_df.isna().sum().sum())

Permutation NaNs: 0
SHAP NaNs: 0
Native NaNs: 0


### Rank agreement


In [ ]:
print(global_results["rank_agreement"])

display(global_results["rank_compare"].head())

{'Native_vs_SHAP': np.float64(0.9808695652173912), 'Native_vs_Permutation': np.float64(0.728142739608856), 'SHAP_vs_Permutation': np.float64(0.7407569208804551)}


,Feature,Importance_x,Method,NativeRank,MeanAbsSHAP,MeanSHAP,SHAPRank,Importance_y,Std,PermutationRank
0,sc,0.172794,tree_importance,1.0,0.082862,-0.015533,2.0,0.006681,0.003167,2.0
1,sg,0.172209,tree_importance,2.0,0.093746,-0.004480,1.0,0.011378,0.006472,1.0
2,hemo,0.151966,tree_importance,3.0,0.078469,-0.003660,3.0,0.000074,0.000873,15.5
3,pcv,0.137196,tree_importance,4.0,0.074007,-0.001781,4.0,0.003311,0.002984,3.0
4,al,0.073456,tree_importance,5.0,0.043660,-0.000041,5.0,0.000459,0.000434,5.0


### Representative cases


In [ ]:
cases = local_results["cases"]

display(
    cases[
        ["CaseType", "y_true", "y_pred", "y_prob"]
    ]
)

,CaseType,y_true,y_pred,y_prob
0,TP,1,1,1.000
2,TP,1,1,1.000
5,TP,1,1,1.000
6,TP,1,1,1.000
10,TP,1,1,1.000
11,TP,1,1,1.000
24,TP,1,1,1.000
25,TP,1,1,1.000
28,TP,1,1,1.000
35,TP,1,1,1.000


### Local explanations and seed stability


In [ ]:
lime_df = local_results["lime"]
lime_summary_df = local_results["lime_summary"]
lime_seed_stability_df = local_results["lime_seed_stability"]

print("Repeated LIME rows:", len(lime_df))
print("Cases by category:")
display(local_results["cases"].groupby("CaseType").size().rename("N cases"))
print("Seeds per case:")
display(lime_df.groupby(["CaseType", "CaseIndex"])["LIMESeed"].nunique().describe())
print("Aggregated LIME summary:")
display(lime_summary_df.head(20))
print("LIME seed-stability summary:")
display(
    lime_seed_stability_df.groupby("CaseType")[[
        "MeanPairwiseSpearman", "MeanTopKOverlap",
        "MeanRuleWeightSD", "MeanRuleSignConsistency"
    ]].mean().round(3)
)

Repeated LIME rows: 3200
Cases by category:


CaseType
Borderline    10
FN             1
FP             1
TN            10
TP            10
Name: N cases, dtype: int64

Seeds per case:


count    32.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max      20.0
Name: LIMESeed, dtype: float64

Aggregated LIME summary:


,CaseType,Rule,MeanWeight,StdWeight,MeanAbsWeight,MedianWeight,PositiveWeightFraction,NExplanations,NCases,NSeeds,SelectionFrequency
0,Borderline,pcv > 45.00,-0.095189,0.004596,0.095189,-0.095193,0.0,120,6,20,0.600
1,Borderline,hemo > 15.00,-0.094963,0.003770,0.094963,-0.095264,0.0,60,3,20,0.300
2,Borderline,sg > 1.02,-0.090826,0.005567,0.090826,-0.089582,0.0,20,1,20,0.100
3,Borderline,sc <= 0.97,-0.082467,0.004022,0.082467,-0.082537,0.0,40,2,20,0.200
4,Borderline,htn <= 0.00,-0.071410,0.003535,0.071410,-0.071655,0.0,200,10,20,1.000
5,Borderline,sc > 3.20,0.071157,0.002812,0.071157,0.071475,1.0,20,1,20,0.100
6,Borderline,sg <= 1.01,0.070673,0.003296,0.070673,0.070409,1.0,40,2,20,0.200
7,Borderline,10.30 < hemo <= 12.70,0.067375,0.003969,0.067375,0.067491,1.0,80,4,20,0.400
8,Borderline,1.35 < sc <= 3.20,0.054837,0.003399,0.054837,0.054785,1.0,78,4,20,0.390
9,Borderline,al <= 0.00,-0.054311,0.003271,0.054311,-0.054059,0.0,80,4,20,0.400


LIME seed-stability summary:


,MeanPairwiseSpearman,MeanTopKOverlap,MeanRuleWeightSD,MeanRuleSignConsistency
CaseType,,,,
Borderline,0.972,0.957,0.008,0.848
FN,0.989,0.937,0.006,0.833
FP,0.990,0.853,0.004,1.000
TN,0.884,0.939,0.007,0.910
TP,0.820,0.866,0.007,0.848


### Bootstrap-refit stability


In [ ]:
if stability_results is not None:
    display(stability_results["raw"].head())
    print(stability_results["summary"])

,Feature,Importance,ImportanceStd,Bootstrap,BootstrapSampleSeed,Rank,InTopK
0,age,0.002214,0.001240,1,383329927,18.0,False
1,bp,0.007135,0.001402,1,383329927,12.0,False
2,sg,0.154430,0.021162,1,383329927,1.0,True
3,al,0.070409,0.006099,1,383329927,5.0,False
4,su,0.006969,0.001639,1,383329927,14.0,False


   Feature  MeanImportance  StdImportance  MeanRank   StdRank  TopKFrequency
20      sg        0.137657       0.030338     1.350  0.641573           1.00
14     pcv        0.099073       0.016766     2.650  0.914253           0.82
19      sc        0.093159       0.023374     2.940  1.245760           0.69
10    hemo        0.078579       0.021927     3.770  1.332234           0.35
1       al        0.061556       0.014982     5.070  1.350309           0.11
11     htn        0.052977       0.009216     6.000  0.910100           0.00
9       dm        0.040726       0.016337     7.180  1.493285           0.03
5      bgr        0.037263       0.009299     7.320  0.952190           0.00
18      rc        0.018644       0.007315     9.990  1.459867           0.00
3    appet        0.016723       0.006314    10.370  1.418884           0.00
7       bu        0.014117       0.005344    11.000  1.497473           0.00
17     rbc        0.012254       0.005905    11.880  2.314993           0.00

### Exported artefacts


In [ ]:

out_dir = Path(config["output_dir"]).resolve()

print("Output directory:", out_dir.relative_to(PROJECT_ROOT))

for path in sorted(out_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(out_dir))

Output directory:

 results/ckd
bundle\metadata.json
bundle\model.pkl
bundle\X_test.csv
bundle\X_train.csv
bundle\y_test.csv
bundle\y_train.csv
execution_times.json
lime_explanations.csv
lime_seed_stability.csv
lime_summary.csv
native_importance.csv
permutation_importance.csv
rank_agreement\Native_vs_Permutation.json
rank_agreement\Native_vs_SHAP.json
rank_agreement\SHAP_vs_Permutation.json
rank_compare.csv
representative_cases.csv
shap_importance.csv
stability.csv
stability_rank.csv
stability_summary.csv


In [ ]:
print(
    experiment.state.results["local"]["cases"].columns.tolist()
)

display(
    experiment.state.results["local"]["cases"]
)

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,appet,pe,ane,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,71.0,60.0,1.015,4.0,0.0,1.0,1.0,0.0,0.0,118.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,0,1
2,52.0,90.0,1.015,4.0,3.0,1.0,0.0,0.0,0.0,224.0,...,0.0,0.0,1.0,1,1,1.000,0.500,TP,2,2
5,56.0,90.0,1.015,2.0,0.0,0.0,0.0,0.0,0.0,129.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,5,3
6,66.0,70.0,1.015,2.0,5.0,1.0,1.0,0.0,0.0,447.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,6,4
10,68.0,90.0,1.010,0.0,0.0,1.0,1.0,0.0,0.0,238.0,...,1.0,0.0,0.0,1,1,1.000,0.500,TP,10,5
11,59.0,80.0,1.010,1.0,0.0,0.0,1.0,0.0,0.0,303.0,...,1.0,0.0,0.0,1,1,1.000,0.500,TP,11,6
24,64.0,60.0,1.010,4.0,1.0,0.0,0.0,0.0,1.0,239.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,24,7
25,73.0,80.0,1.020,2.0,0.0,0.0,0.0,0.0,0.0,253.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,25,8
28,54.0,60.0,1.015,3.0,2.0,0.0,0.0,0.0,0.0,352.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,28,9
35,69.0,70.0,1.010,3.0,4.0,1.0,0.0,0.0,0.0,264.0,...,0.0,1.0,0.0,1,1,1.000,0.500,TP,35,10


In [ ]:
assert "CaseIndex" in experiment.state.results["local"]["cases"].columns
print("CKD CaseIndex exists in memory.")

CKD CaseIndex exists in memory.

In [ ]:
ckd_cases = pd.read_csv(
    Path("../../results/ckd/representative_cases.csv")
)

print(ckd_cases.columns.tolist())
display(ckd_cases)

['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wc', 'rc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'y_true', 'y_pred', 'y_prob', 'distance', 'CaseType', 'CaseIndex', 'CaseRank']


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,appet,pe,ane,y_true,y_pred,y_prob,distance,CaseType,CaseIndex,CaseRank
0,71.0,60.0,1.015,4.0,0.0,1.0,1.0,0.0,0.0,118.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,0,1
1,52.0,90.0,1.015,4.0,3.0,1.0,0.0,0.0,0.0,224.0,...,0.0,0.0,1.0,1,1,1.000,0.500,TP,2,2
2,56.0,90.0,1.015,2.0,0.0,0.0,0.0,0.0,0.0,129.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,5,3
3,66.0,70.0,1.015,2.0,5.0,1.0,1.0,0.0,0.0,447.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,6,4
4,68.0,90.0,1.010,0.0,0.0,1.0,1.0,0.0,0.0,238.0,...,1.0,0.0,0.0,1,1,1.000,0.500,TP,10,5
5,59.0,80.0,1.010,1.0,0.0,0.0,1.0,0.0,0.0,303.0,...,1.0,0.0,0.0,1,1,1.000,0.500,TP,11,6
6,64.0,60.0,1.010,4.0,1.0,0.0,0.0,0.0,1.0,239.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,24,7
7,73.0,80.0,1.020,2.0,0.0,0.0,0.0,0.0,0.0,253.0,...,0.0,0.0,0.0,1,1,1.000,0.500,TP,25,8
8,54.0,60.0,1.015,3.0,2.0,0.0,0.0,0.0,0.0,352.0,...,1.0,1.0,0.0,1,1,1.000,0.500,TP,28,9
9,69.0,70.0,1.010,3.0,4.0,1.0,0.0,0.0,0.0,264.0,...,0.0,1.0,0.0,1,1,1.000,0.500,TP,35,10


In [ ]:
results_dir = Path(config["output_dir"]).resolve()
stability_file = results_dir / "stability.csv"

print("Reading:", stability_file)

stability = pd.read_csv(stability_file)

importance_by_bootstrap = stability.pivot(
    index="Bootstrap",
    columns="Feature",
    values="Importance",
)

print("Bootstrap refits:", len(importance_by_bootstrap))
print(
    "Distinct importance profiles:",
    len(importance_by_bootstrap.drop_duplicates()),
)

display(importance_by_bootstrap)

Reading: results/ckd\stability.csv
Bootstrap refits: 100
Distinct importance profiles: 100


Feature,age,al,ane,appet,ba,bgr,bp,bu,cad,dm,...,pcv,pe,pot,rbc,rc,sc,sg,sod,su,wc
Bootstrap,,,,,,,,,,,,,,,,,,,,,
1,0.002214,0.070409,0.002789,0.015326,0.000161,0.041102,0.007135,0.013055,0.000000,0.058448,...,0.097208,0.005758,0.001362,0.006516,0.020535,0.120855,0.154430,0.007073,0.006969,0.001817
2,0.001023,0.048208,0.001585,0.014915,0.000042,0.048603,0.006240,0.013949,0.000576,0.020957,...,0.095392,0.010343,0.000594,0.018328,0.015290,0.107369,0.143921,0.004665,0.004475,0.001629
3,0.002331,0.071885,0.002033,0.017450,0.000165,0.041282,0.005824,0.010399,0.000000,0.039355,...,0.093788,0.003791,0.001505,0.010267,0.022353,0.077283,0.119990,0.003581,0.007282,0.003175
4,0.000983,0.057178,0.001293,0.027701,0.000091,0.030759,0.008145,0.021458,0.000408,0.042301,...,0.098588,0.002872,0.002124,0.012274,0.009305,0.066396,0.167148,0.003850,0.004732,0.001591
5,0.002151,0.076431,0.002896,0.026114,0.000473,0.055154,0.005865,0.019814,0.000520,0.063159,...,0.091280,0.008396,0.001135,0.009127,0.010212,0.105959,0.139711,0.005383,0.006551,0.000429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,0.002640,0.054161,0.001709,0.011206,0.000269,0.030218,0.006265,0.014803,0.000160,0.033041,...,0.102571,0.012493,0.002199,0.013063,0.017564,0.088539,0.109526,0.003814,0.005374,0.002200
97,0.002628,0.067521,0.001945,0.008240,0.000260,0.032141,0.004474,0.015520,0.000582,0.036291,...,0.090451,0.005730,0.002214,0.016444,0.019845,0.109600,0.169657,0.002562,0.006798,0.004271
98,0.001556,0.049007,0.001822,0.018424,0.000081,0.047307,0.011715,0.021963,0.000151,0.044904,...,0.095434,0.005372,0.001409,0.015575,0.016771,0.109986,0.129865,0.001883,0.004433,0.002690


## Interpretation

Native importance and SHAP were closely aligned, and CKD produced the highest complete-ranking persistence across bootstrap refits. The dataset also produced near-perfect held-out discrimination. These results remain an internal evaluation of a small, unusually separable dataset; duplicate checks were negative, but external validity is not established.
